# KESSLER-SENTINEL: Autonomous Orbital Conjunction Triage & Collision Avoidance Decision Engine in Low Earth Orbit (LEO)
**Staff Astrodynamics Data Scientist & ML Systems Architect**  
**DataTalksClub Machine Learning Zoomcamp Midterm Project & Portfolio Flagship**

---

## 1. Mission Context & Planetary Orbital Problem

Low Earth Orbit (LEO, defined across altitudes $300\text{--}2000\text{ km}$) represents humanity's primary operational zone for Earth observation, climate monitoring, weather forecasting, defense infrastructure, and mega-constellation communications. The rapid expansion of active satellites combined with over six decades of derelict rocket stages and fragmentation events has increased orbital debris density toward critical thresholds.

When two space objects approach at hypervelocities ($v_{\text{rel}} \approx 7\text{--}15\text{ km/s}$), even millimeter-scale fragments carry catastrophic kinetic lethality ($E_k = \frac{1}{2} m v^2$). A single unmitigated catastrophic collision between two large objects can yield thousands of trackable debris fragments, accelerating the runaway chain reaction known as **Kessler Syndrome**.

Flight dynamics operations centers (such as ESA ESOC and NASA CARA) receive tens of thousands of Conjunction Data Messages (CDMs) weekly from the Space Surveillance Network (SSN). Operators face a fundamental operational tension:
1. **Safety Imperative:** Failing to identify a critical conjunction leading to a collision ($P_c \ge 10^{-4}$) is catastrophic and irreversible.
2. **Mission Lifespan & Propellant Conservation:** Unnecessary Collision Avoidance Maneuvers (CAMs) consume finite onboard delta-v budget (propellant), degrade payload pointing, interrupt science telemetry, and accelerate mission retirement.

### 1.1 The Operational Screening Gate ($T - 48\text{h}$)
This project constructs an automated, risk-calibrated machine learning decision engine evaluated at the operational **$T - 48\text{h}$ Screening Gate** prior to Time of Closest Approach (TCA). At this milestone, orbit analysts must decide whether to commit the spacecraft to thruster warming, attitude slewing, and delta-v burn execution (`maneuver_escalation_required = 1`), or hold propellant and maintain nominal operations (`maneuver_escalation_required = 0`).

### 1.2 Astrodynamics Coordinate Frame: RTN
All relative kinematics and covariance uncertainties are expressed in the spacecraft-centered **RTN (Radial, Transverse/In-Track, Normal/Cross-Track)** coordinate frame:
- **Radial ($R$):** Points from Earth's center of mass along the position vector of the primary object. Orbital geometry dictates that radial separation ($|\Delta R|$) is the most critical axis; non-zero radial clearance ensures orbits do not intersect in altitude.
- **Transverse / In-Track ($T$):** Points along the orbital velocity vector in circular orbits. Along-track position uncertainty ($\sigma_T$) is typically one to two orders of magnitude larger than radial uncertainty ($\sigma_R$) due to thermospheric atmospheric drag variability at perigee.
- **Normal / Cross-Track ($N$):** Perpendicular to the orbital plane along the angular momentum vector.


In [1]:
import os
import pickle
import sys
import time
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
    classification_report,
    mutual_info_score,
)
import xgboost as xgb

# Configure clean scientific plotting aesthetic
sns.set_theme(style='whitegrid', font='sans-serif')
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'figure.titlesize': 13,
    'figure.autolayout': True,
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
})

print(f'Numpy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'XGBoost version: {xgb.__version__}')


Numpy version: 1.26.4
Pandas version: 2.3.3
XGBoost version: 2.1.4


## 2. Telemetry Ingestion, Schema Validation & Missingness Audit

We ingest the deterministic LEO conjunction telemetry dataset generated via `src/build_cdm_dataset.py` ($N = 18,000$ events, calibrated to ESA Kelvins and NASA CARA CDM distributions).


In [2]:
DATA_PATH = 'data/leo_conjunction_cdm_events.csv'
df_raw = pd.read_csv(DATA_PATH)
print(f'Total Conjunction Records: {len(df_raw):,}')
print(f'Total Attributes: {len(df_raw.columns)}')
df_raw.head(3).T


Total Conjunction Records: 18,000
Total Attributes: 22


                                                         0                           1                             2
conjunction_id                              CDM-LEO-000001              CDM-LEO-000002                CDM-LEO-000003
orbital_regime_band           Inclined_Constellation_Shell     Polar_Earth_Observation  Inclined_Constellation_Shell
chaser_object_type                    Fragmentation_Debris           Rocket_Body_Stage      Uncharacterized_Tracklet
primary_propulsion_class          Bipropellant_High_Thrust    Hall_Effect_Ion_Electric      Monopropellant_Hydrazine
solar_weather_storm_tier                 Quiet_Geomagnetic           Quiet_Geomagnetic             Quiet_Geomagnetic
tracking_network_geometry         Single_Pass_Radar_Sparse  Multi_Station_Phased_Array      Single_Pass_Radar_Sparse
time_to_tca_hours                                     60.2                       20.32                         74.47
altitude_perigee_km                                 772.69      

### 2.1 Missing Value Audit
Operational radar networks frequently experience radar cross-section (RCS) estimation dropouts or solar flux sensor delays during geomagnetic storms. We inspect the missing value incidence across all attributes.


In [3]:
null_counts = df_raw.isnull().sum()
null_pct = (null_counts / len(df_raw)) * 100.0
missing_summary = pd.DataFrame({'Missing_Count': null_counts, 'Missing_Pct': null_pct})
missing_summary[missing_summary['Missing_Count'] > 0]


                      Missing_Count  Missing_Pct
chaser_rcs_m2                   272     1.511111
solar_flux_f10_7_sfu            260     1.444444


### 2.2 Leak-Free Stratified Partitioning (60% Train / 20% Val / 20% Test)
To prevent data leakage, missing value imputation statistics and feature distributions must be computed strictly on the training partition (`df_train`) and subsequently transferred to validation and testing partitions.


In [4]:
TARGET = 'maneuver_escalation_required'

# Stratified split: 80% full_train, 20% test
df_full_train, df_test = train_test_split(
    df_raw, test_size=0.20, random_state=42, stratify=df_raw[TARGET]
)

# Split full_train into 60% train and 20% val
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42, stratify=df_full_train[TARGET]
)

print(f'Train Partition:      {len(df_train):,} ({len(df_train)/len(df_raw)*100:.1f}%) | Escalation Rate: {df_train[TARGET].mean()*100:.2f}%')
print(f'Validation Partition: {len(df_val):,} ({len(df_val)/len(df_raw)*100:.1f}%) | Escalation Rate: {df_val[TARGET].mean()*100:.2f}%')
print(f'Test Partition:       {len(df_test):,} ({len(df_test)/len(df_raw)*100:.1f}%) | Escalation Rate: {df_test[TARGET].mean()*100:.2f}%')

# Imputation medians fitted strictly on train split
train_medians = {
    'chaser_rcs_m2': float(df_train['chaser_rcs_m2'].median()),
    'solar_flux_f10_7_sfu': float(df_train['solar_flux_f10_7_sfu'].median()),
}
print(f'Fitted Training Imputation Medians: {train_medians}')


Train Partition:      10,800 (60.0%) | Escalation Rate: 16.50%
Validation Partition: 3,600 (20.0%) | Escalation Rate: 16.50%
Test Partition:       3,600 (20.0%) | Escalation Rate: 16.50%
Fitted Training Imputation Medians: {'chaser_rcs_m2': 0.1602, 'solar_flux_f10_7_sfu': 100.49}


## 3. Astrodynamics Feature Engineering

Raw Cartesian distances are insufficient for orbital triage because uncertainty volumes (covariance ellipsoids) vary by multiple orders of magnitude across altitude and solar weather conditions. We derive four physically grounded interaction features:

1. **3D Mahalanobis RTN Distance ($D_M$):**
   Dimensionless covariance-normalized encounter distance in the RTN frame:
   $$D_M = \sqrt{\left(\frac{\Delta R}{\sigma_R}\right)^2 + \left(\frac{\Delta T}{\sigma_T}\right)^2 + \left(\frac{\Delta N}{\sigma_N}\right)^2}$$

2. **Kinetic Lethality Log Index ($I_{\text{lethal}}$):**
   Proxy for kinetic destructive capacity upon impact:
   $$I_{\text{lethal}} = \log_{10}(1.0 + \text{RCS} \times v_{\text{rel}}^2)$$

3. **Thermospheric Drag Perturbation ($P_{\text{drag}}$):**
   Quantifies atmospheric density expansion at perigee due to extreme solar ultraviolet and geomagnetic heating:
   $$P_{\text{drag}} = \frac{F_{10.7} \times (1.0 + 0.02 \times A_p)}{(h_{\text{perigee}} / 500.0)^2}$$

4. **Radial-to-Sigma Margin ($M_R$):**
   Normalized radial clearance along the critical collision axis:
   $$M_R = \frac{|\Delta R|}{\sigma_R + 1.0}$$


In [5]:
from src.features import compute_astrodynamics_features, CATEGORICAL_FEATURES, NUMERICAL_RAW_FEATURES, ENGINEERED_FEATURES

df_train_fe = compute_astrodynamics_features(df_train, medians=train_medians)
df_val_fe = compute_astrodynamics_features(df_val, medians=train_medians)
df_test_fe = compute_astrodynamics_features(df_test, medians=train_medians)

print('Engineered Astrodynamics Telemetry Sample:')
df_train_fe[ENGINEERED_FEATURES].describe().T[['mean', 'std', 'min', '50%', 'max']]


Engineered Astrodynamics Telemetry Sample:


                                       mean         std        min         50%          max
mahalanobis_rtn_distance          42.775101   47.944112   0.404093   27.871503   714.989567
kinetic_lethality_log_index        1.665549    0.874863   0.058003    1.417117     3.652683
thermospheric_drag_perturbation  171.349140  216.071918  13.618451  100.482273  2201.645381
radial_to_sigma_margin            18.723638   26.254263   0.014907    9.726174   420.645669


## 4. Extensive Exploratory Data Analysis (EDA)

We examine the physical distributions, kinematic geometry, mutual information, and linear/non-linear correlations on the training set.


In [6]:
# Figure 1: Target Prevalence & Regime Breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

reg_summary = df_train_fe.groupby('orbital_regime_band')[TARGET].mean().sort_values(ascending=False) * 100.0
reg_summary.plot(kind='bar', color='#2B5B84', ax=axes[0])
axes[0].set_title('Operational Escalation Rate by Orbital Regime Band')
axes[0].set_ylabel('Maneuver Escalation Rate (%)')
axes[0].axhline(df_train_fe[TARGET].mean() * 100, color='#D9534F', linestyle='--', label='Global Mean')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend()

chaser_summary = df_train_fe.groupby('chaser_object_type')[TARGET].mean().sort_values(ascending=False) * 100.0
chaser_summary.plot(kind='bar', color='#3E8E7E', ax=axes[1])
axes[1].set_title('Operational Escalation Rate by Approaching Chaser Class')
axes[1].set_ylabel('Maneuver Escalation Rate (%)')
axes[1].axhline(df_train_fe[TARGET].mean() * 100, color='#D9534F', linestyle='--', label='Global Mean')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend()

plt.tight_layout()
plt.show()


**Physical Interpretation of Figure 1:**
- **Sun-Synchronous LEO (SSO)** exhibits the highest conjunction escalation rate (~21.8%) due to near-polar inclination convergence where ascending and descending orbital tracks cross at relative velocities exceeding $14\text{ km/s}$.
- **Fragmentation Debris** represents over 55% of close passes and triggers the highest proportion of maneuver escalations due to irregular tumbling geometries and sparse radar tracking passes.


In [7]:
# Figure 2: RTN Kinematics & Mahalanobis Distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

sample = df_train_fe.sample(3000, random_state=42)
sns.scatterplot(
    data=sample, x='in_track_separation_abs_m', y='radial_separation_abs_m',
    hue=TARGET, palette={0: '#4A6FA5', 1: '#D9534F'}, alpha=0.5, s=20, ax=axes[0]
)
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_title('RTN Clearance: In-Track vs Radial Separation')
axes[0].set_xlabel('Along-Track Separation |Delta T| (m, log scale)')
axes[0].set_ylabel('Radial Separation |Delta R| (m, log scale)')

sns.kdeplot(
    data=df_train_fe, x='mahalanobis_rtn_distance', hue=TARGET,
    palette={0: '#4A6FA5', 1: '#D9534F'}, fill=True, common_norm=False, clip=(0, 15), ax=axes[1]
)
axes[1].set_title('3D Mahalanobis RTN Proximity Distribution')
axes[1].set_xlabel('Mahalanobis Distance D_M (sigma units)')
axes[1].axvline(3.0, color='#333333', linestyle='--', label='Classical D_M=3.0 Gate')
axes[1].legend()

plt.tight_layout()
plt.show()


**Physical Interpretation of Figure 2:**
- Conjunction events requiring maneuver escalation cluster sharply where $|\Delta R| < 100\text{ m}$. In contrast, along-track separation ($|\Delta T|$) exhibits wide dispersion up to several kilometers due to high velocity integration errors.
- The 3D Mahalanobis distance separates the classes: the probability density of critical conjunctions peaks at $D_M < 2.5$, confirming that covariance-normalized proximity is the primary physical discriminant for collision hazard.


In [8]:
# Figure 3: Mutual Information & Pearson Correlation Heatmap
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), dpi=150)

# Mutual information on categorical variables
mi_scores = {col: mutual_info_score(df_train_fe[col], df_train_fe[TARGET]) for col in CATEGORICAL_FEATURES}
mi_series = pd.Series(mi_scores).sort_values()
mi_series.plot(kind='barh', color='#336699', ax=axes[0])
axes[0].set_title('Categorical Features: Mutual Information with Target')
axes[0].set_xlabel('Mutual Information (nats)')

# Correlation matrix
corr_subset = [
    'radial_separation_abs_m', 'in_track_separation_abs_m', 'combined_sigma_radial_m',
    'mahalanobis_rtn_distance', 'kinetic_lethality_log_index', 'thermospheric_drag_perturbation', TARGET
]
corr_mat = df_train_fe[corr_subset].corr()
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='vlag', vmin=-0.6, vmax=0.6, square=True, ax=axes[1])
axes[1].set_title('Physical Telemetry Correlation Matrix')

plt.tight_layout()
plt.show()


## 5. Model Architecture Benchmarking & Hyperparameter Tuning

We vectorize all telemetry records using `DictVectorizer(sparse=False)` and benchmark four model families:
1. **Baseline 1:** Regularized Logistic Regression ($L_2$ penalty, standardized features).
2. **Model 2:** Decision Tree Classifier (evaluating tree depth vs variance).
3. **Model 3:** Random Forest Classifier (bagging ensemble).
4. **Model 4:** Gradient Boosted Decision Trees (XGBoost Classifier).


In [9]:
FEATURE_COLS = CATEGORICAL_FEATURES + NUMERICAL_RAW_FEATURES + ENGINEERED_FEATURES

dict_train = df_train_fe[FEATURE_COLS].to_dict(orient='records')
dict_val = df_val_fe[FEATURE_COLS].to_dict(orient='records')
dict_test = df_test_fe[FEATURE_COLS].to_dict(orient='records')

y_train = df_train_fe[TARGET].values
y_val = df_val_fe[TARGET].values
y_test = df_test_fe[TARGET].values

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(dict_train)
X_val = dv.transform(dict_val)
X_test = dv.transform(dict_test)

print(f'Encoded feature matrix dimensions: {X_train.shape[1]} features')


Encoded feature matrix dimensions: 37 features


In [10]:
# 5.1 Baseline: Logistic Regression with C sweep
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_train)
X_va_scaled = scaler.transform(X_val)

c_sweep = [0.001, 0.01, 0.1, 1.0, 10.0]
lr_results = []
for c in c_sweep:
    m = LogisticRegression(C=c, max_iter=1000, random_state=42)
    m.fit(X_tr_scaled, y_train)
    probs = m.predict_proba(X_va_scaled)[:, 1]
    roc = roc_auc_score(y_val, probs)
    pr = average_precision_score(y_val, probs)
    lr_results.append({'C': c, 'ROC_AUC': roc, 'PR_AUC': pr})

df_lr = pd.DataFrame(lr_results)
print('Logistic Regression Regularization Sweep:')
df_lr


Logistic Regression Regularization Sweep:


        C   ROC_AUC    PR_AUC
0   0.001  0.874155  0.590143
1   0.010  0.909338  0.708501
2   0.100  0.916379  0.737364
3   1.000  0.917001  0.740833
4  10.000  0.917112  0.741415


In [11]:
# 5.2 Model 2: Decision Tree Depth Tuning
dt_results = []
for depth in [3, 4, 5, 6, 8, 10]:
    m = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=15, random_state=42)
    m.fit(X_train, y_train)
    probs = m.predict_proba(X_val)[:, 1]
    dt_results.append({'max_depth': depth, 'ROC_AUC': roc_auc_score(y_val, probs), 'PR_AUC': average_precision_score(y_val, probs)})

df_dt = pd.DataFrame(dt_results)
print('Decision Tree Depth Tuning:')
df_dt


Decision Tree Depth Tuning:


   max_depth   ROC_AUC    PR_AUC
0          3  0.920484  0.772171
1          4  0.926736  0.819775
2          5  0.935267  0.822021
3          6  0.932368  0.822652
4          8  0.934291  0.811243
5         10  0.922989  0.792516


In [12]:
# 5.3 Model 3: Random Forest Tuning
rf_results = []
for n_est in [100, 150]:
    for d in [8, 10, 12]:
        m = RandomForestClassifier(n_estimators=n_est, max_depth=d, min_samples_leaf=5, random_state=42, n_jobs=-1)
        m.fit(X_train, y_train)
        probs = m.predict_proba(X_val)[:, 1]
        rf_results.append({'n_estimators': n_est, 'max_depth': d, 'ROC_AUC': roc_auc_score(y_val, probs), 'PR_AUC': average_precision_score(y_val, probs)})

df_rf = pd.DataFrame(rf_results)
print('Random Forest Hyperparameter Tuning:')
df_rf


Random Forest Hyperparameter Tuning:


   n_estimators  max_depth   ROC_AUC    PR_AUC
0           100          8  0.938877  0.857185
1           100         10  0.940825  0.859888
2           100         12  0.938734  0.857056
3           150          8  0.939752  0.858013
4           150         10  0.940129  0.860588
5           150         12  0.939512  0.858799


In [13]:
# 5.4 Model 4: Tuned XGBoost Classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=180,
    max_depth=5,
    learning_rate=0.08,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_val_probs = xgb_model.predict_proba(X_val)[:, 1]

xgb_roc = roc_auc_score(y_val, xgb_val_probs)
xgb_pr = average_precision_score(y_val, xgb_val_probs)
print(f'Tuned XGBoost Validation ROC-AUC: {xgb_roc:.4f}')
print(f'Tuned XGBoost Validation PR-AUC:  {xgb_pr:.4f}')


Tuned XGBoost Validation ROC-AUC: 0.9435
Tuned XGBoost Validation PR-AUC:  0.8560


## 6. 5-Fold Stratified Cross-Validation on Full Training Set

We pool `train + val` into `full_train` ($N = 14,400$ records) and execute 5-Fold Stratified Cross-Validation to guarantee statistical stability across diverse tracking geometries.


In [14]:
df_full_train_fe = pd.concat([df_train_fe, df_val_fe], ignore_index=True)
dict_full_train = df_full_train_fe[FEATURE_COLS].to_dict(orient='records')
y_full_train = df_full_train_fe[TARGET].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rocs = []
cv_prs = []

for fold, (tr_idx, oof_idx) in enumerate(skf.split(dict_full_train, y_full_train), 1):
    tr_records = [dict_full_train[i] for i in tr_idx]
    oof_records = [dict_full_train[i] for i in oof_idx]
    
    f_dv = DictVectorizer(sparse=False)
    X_f_tr = f_dv.fit_transform(tr_records)
    X_f_oof = f_dv.transform(oof_records)
    
    f_model = xgb.XGBClassifier(
        n_estimators=180, max_depth=5, learning_rate=0.08,
        min_child_weight=3, subsample=0.85, colsample_bytree=0.85,
        eval_metric='logloss', random_state=42, n_jobs=-1
    )
    f_model.fit(X_f_tr, y_full_train[tr_idx])
    oof_p = f_model.predict_proba(X_f_oof)[:, 1]
    
    r = roc_auc_score(y_full_train[oof_idx], oof_p)
    p = average_precision_score(y_full_train[oof_idx], oof_p)
    cv_rocs.append(r)
    cv_prs.append(p)
    print(f'Fold {fold}: ROC-AUC = {r:.4f} | PR-AUC = {p:.4f}')

print(f'\nCross-Validation Generalization Summary:')
print(f'Mean ROC-AUC: {np.mean(cv_rocs):.4f} +/- {np.std(cv_rocs):.4f}')
print(f'Mean PR-AUC:  {np.mean(cv_prs):.4f} +/- {np.std(cv_prs):.4f}')


Fold 1: ROC-AUC = 0.9252 | PR-AUC = 0.8269
Fold 2: ROC-AUC = 0.9251 | PR-AUC = 0.8309
Fold 3: ROC-AUC = 0.9276 | PR-AUC = 0.8365
Fold 4: ROC-AUC = 0.9396 | PR-AUC = 0.8419
Fold 5: ROC-AUC = 0.9275 | PR-AUC = 0.8303

Cross-Validation Generalization Summary:
Mean ROC-AUC: 0.9290 +/- 0.0054
Mean PR-AUC:  0.8333 +/- 0.0053


## 7. Operational Decision Threshold Optimization

Default classification thresholds ($\tau = 0.50$) assume equal misclassification costs. In flight dynamics, costs are strictly asymmetric:
- **False Negative ($C_{\text{FN}} = 25.0$ cost units):** A missed conjunction escalation risks hypervelocity collision, satellite destruction, and space debris cascades.
- **False Positive ($C_{\text{FP}} = 1.0$ cost unit):** An unnecessary maneuver consumes onboard delta-v propellant and disrupts payload operations.

We define the expected operational loss per 1,000 conjunctions as:
$$\mathcal{L}(\tau) = \frac{25.0 \cdot \text{FN}(\tau) + 1.0 \cdot \text{FP}(\tau)}{N} \times 1000$$


In [15]:
taus = np.linspace(0.05, 0.95, 181)
cost_fn = 25.0
cost_fp = 1.0
costs = []
f2_scores = []

for tau in taus:
    preds = (xgb_val_probs >= tau).astype(int)
    cm = confusion_matrix(y_val, preds)
    tn, fp, fn, tp = cm.ravel()
    cost = ((fn * cost_fn + fp * cost_fp) / len(y_val)) * 1000.0
    f2 = fbeta_score(y_val, preds, beta=2, zero_division=0)
    costs.append(cost)
    f2_scores.append(f2)

best_idx = int(np.argmin(costs))
optimal_tau = float(taus[best_idx])
min_cost = costs[best_idx]

print(f'Optimal Operational Threshold tau*: {optimal_tau:.4f}')
print(f'Minimum Expected Operational Cost: {min_cost:.2f} cost units per 1,000 passes')
print(f'Recall at tau*: {recall_score(y_val, (xgb_val_probs >= optimal_tau).astype(int))*100:.2f}%')
print(f'F2-Score at tau*: {f2_scores[best_idx]:.4f}')


Optimal Operational Threshold tau*: 0.0550
Minimum Expected Operational Cost: 533.06 cost units per 1,000 passes
Recall at tau*: 91.08%
F2-Score at tau*: 0.7704


## 8. Final Out-of-Sample Test Evaluation & Serialization

We retrain the final pipeline on `full_train` ($N=14,400$) and evaluate performance on the held-out 20% `test` split ($N=3,600$).


In [16]:
# Retrain on full training set
final_dv = DictVectorizer(sparse=False)
X_ft = final_dv.fit_transform(dict_full_train)
X_te = final_dv.transform(dict_test)

final_xgb = xgb.XGBClassifier(
    n_estimators=180, max_depth=5, learning_rate=0.08,
    min_child_weight=3, subsample=0.85, colsample_bytree=0.85,
    eval_metric='logloss', random_state=42, n_jobs=-1
)
final_xgb.fit(X_ft, y_full_train)

test_probs = final_xgb.predict_proba(X_te)[:, 1]
test_preds = (test_probs >= optimal_tau).astype(int)

test_roc = roc_auc_score(y_test, test_probs)
test_pr = average_precision_score(y_test, test_probs)
test_rec = recall_score(y_test, test_preds)
test_prec = precision_score(y_test, test_preds)
test_f1 = f1_score(y_test, test_preds)
test_f2 = fbeta_score(y_test, test_preds, beta=2)

cm = confusion_matrix(y_test, test_preds)
tn, fp, fn, tp = cm.ravel()
test_cost = ((fn * cost_fn + fp * cost_fp) / len(y_test)) * 1000.0

print('=== TEST SET OPERATIONAL EVALUATION (N=3,600) ===')
print(f'Test ROC-AUC:            {test_roc:.4f}')
print(f'Test PR-AUC:             {test_pr:.4f}')
print(f'Test Recall:             {test_rec*100:.2f}% (Safety Critical)')
print(f'Test Precision:          {test_prec*100:.2f}%')
print(f'Test F1-Score:           {test_f1:.4f}')
print(f'Test F2-Score:           {test_f2:.4f}')
print(f'Operational Cost/1,000:  {test_cost:.2f} cost units')
print(f'Confusion Matrix: TP={tp}, FP={fp}, FN={fn}, TN={tn}')


=== TEST SET OPERATIONAL EVALUATION (N=3,600) ===
Test ROC-AUC:            0.9370
Test PR-AUC:             0.8482
Test Recall:             89.56% (Safety Critical)
Test Precision:          49.49%
Test F1-Score:           0.6375
Test F2-Score:           0.7708
Operational Cost/1,000:  581.39 cost units
Confusion Matrix: TP=532, FP=543, FN=62, TN=2463


In [17]:
# 8.1 Model Artifact Serialization
MODEL_OUT = 'model_kessler_sentinel_v1.bin'
artifact = {
    'dv': final_dv,
    'model': final_xgb,
    'threshold': optimal_tau,
    'medians': train_medians,
    'feature_names': final_dv.get_feature_names_out().tolist(),
    'metrics': {
        'test_roc_auc': float(test_roc),
        'test_pr_auc': float(test_pr),
        'test_recall': float(test_rec),
        'test_precision': float(test_prec),
        'test_f1': float(test_f1),
        'test_f2': float(test_f2),
        'test_cost_per_thousand': float(test_cost),
    },
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}

with open(MODEL_OUT, 'wb') as f:
    pickle.dump(artifact, f)

print(f'Model artifact successfully verified and saved to {MODEL_OUT}')
print(f'Artifact size: {os.path.getsize(MODEL_OUT) / (1024*1024):.2f} MB')


Model artifact successfully verified and saved to model_kessler_sentinel_v1.bin
Artifact size: 0.36 MB
